# Corporate Innovation Flow

This example demonstrates how **TinySocialNetwork** creates realistic corporate communication dynamics. 
Unlike a plain `TinyWorld` where everyone can talk to everyone, this social network constrains
information flow based on organizational structure — just like a real company.

## Scenario
A mid-size tech company is evaluating ideas for their next product launch. Ideas originate in R&D, 
must be championed through management, and finally evaluated by a market-facing team. We'll observe:

- How information flows (and gets distorted) through hierarchical layers
- Which roles become communication bottlenecks
- How network position influences decision-making

> **Note**: Results are AI-generated and will vary between runs. This is for demonstration purposes only.

In [ ]:
import json
import sys
sys.path.insert(0, '..')

import tinytroupe
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld, TinySocialNetwork, TinySocialNetworkFactory
from tinytroupe.factory import TinyPersonFactory
from tinytroupe.extraction import ResultsReducer
import tinytroupe.control as control

import textwrap

## Step 1: Create the Team

We use `TinyPersonFactory` to generate realistic personas for our company. Each person has a distinct 
role and personality that will influence how they communicate and make decisions.

In [ ]:
company_context = """
A mid-size technology company called 'Meridian Technologies' that develops enterprise software 
solutions. The company has 50 employees and is based in Austin, Texas. The culture values 
innovation and data-driven decision making, but also has typical corporate hierarchy and processes.
"""

factory = TinyPersonFactory(company_context)

# R&D team
rd_lead = factory.generate_person(
    "A senior R&D engineer and team lead, very creative and passionate about new technologies. "
    "Tends to be optimistic about technical feasibility."
)
rd_engineer = factory.generate_person(
    "A junior software engineer in R&D, fresh out of grad school, enthusiastic but sometimes "
    "overly theoretical. Specializes in machine learning."
)

# Management
vp_product = factory.generate_person(
    "A VP of Product who has 15 years of experience. Pragmatic, focused on market fit and ROI. "
    "Sometimes skeptical of purely technical ideas."
)
cto = factory.generate_person(
    "The CTO, a seasoned technology executive who bridges technical and business perspectives. "
    "Values both innovation and practical execution."
)

# Market-facing team
sales_lead = factory.generate_person(
    "A senior sales director who knows customer pain points intimately. Very customer-centric "
    "and sometimes pushes back on features that don't address real customer needs."
)
marketing_mgr = factory.generate_person(
    "A marketing manager who focuses on positioning and messaging. Good at translating "
    "technical capabilities into business value propositions."
)

print("Team created!")
for p in [rd_lead, rd_engineer, vp_product, cto, sales_lead, marketing_mgr]:
    print(f"  - {p.name}")

## Step 2: Build the Corporate Network

Instead of letting everyone talk to everyone (`TinyWorld`), we use `TinySocialNetworkFactory` 
to create a **corporate hierarchy**. This is key — it means:

- R&D can only talk to their manager (CTO) 
- The CTO connects to both R&D and the VP of Product
- The VP of Product connects to the market-facing team
- Sales and marketing can talk to each other and their VP

This mimics how information actually flows in organizations.

In [ ]:
# Create the corporate hierarchy network
agents = [cto, vp_product, rd_lead, rd_engineer, sales_lead, marketing_mgr]

network = TinySocialNetworkFactory.create_corporate_hierarchy(
    "Meridian Technologies",
    agents,
    ceo=cto,
    span_of_control=3,
    relation_name="reports_to"
)

# Also add some cross-functional links that exist in real companies
network.add_relation(rd_lead, vp_product, "cross_functional", 
                     attributes={"description": "Cross-functional collaboration on product requirements"})
network.add_relation(sales_lead, marketing_mgr, "team", 
                     attributes={"description": "Same go-to-market team"})

# Let's see the network structure
print("Network Summary:")
summary = network.get_network_summary()
for key, value in summary.items():
    print(f"  {key}: {value}")

## Step 3: Analyze the Network Structure

Before running the simulation, let's examine the network's structural properties.
These metrics from graph theory tell us about information flow potential.

In [ ]:
print("=== Network Structural Analysis ===\n")

# Degree: how many connections each person has
print("Connections per person (degree):")
for agent, deg in network.degree().items():
    print(f"  {agent.name}: {deg} connections")

print(f"\nNetwork density: {network.density():.2f}")
print(f"  (1.0 = everyone connected to everyone, 0.0 = no connections)")

print(f"\nNetwork is connected: {network.is_connected()}")

# Betweenness centrality: who are the bottlenecks?
print("\nCommunication bottleneck score (betweenness centrality):")
for agent, bc in sorted(network.betweenness_centrality().items(), key=lambda x: -x[1]):
    role = "*** BOTTLENECK ***" if bc > 0.1 else ""
    print(f"  {agent.name}: {bc:.3f} {role}")
    
print(f"\nDiameter (longest shortest path): {network.diameter()}")
print("  This means the most 'hops' a message needs to cross the company is", network.diameter())

## Step 4: Run the Innovation Discussion

Now we seed an idea and let the network run. Only connected agents can communicate,
creating a realistic information cascade through the organization.

In [ ]:
# Seed the idea from R&D
network.broadcast_internal_goal(
    "We need to decide what innovative feature to build for our next product release. "
    "Consider both technical feasibility and market demand."
)

# R&D has a specific idea
rd_lead.think(
    "I have a great idea: we could build an AI-powered document summarization feature "
    "that automatically extracts key insights from enterprise documents. I need to "
    "pitch this to leadership."
)
rd_engineer.think(
    "I've been researching transformer models and I think we could build something "
    "really impressive with the latest open-source LLMs."
)

# Run the simulation for a few rounds - let ideas flow through the hierarchy
network.run(steps=3)

## Step 5: Analyze the Communication Patterns

Let's see what happened — who talked to whom, and how the idea evolved as it 
traveled through the corporate hierarchy.

In [ ]:
# Message analysis
print(f"Total messages exchanged: {network.get_message_count()}\n")

print("Messages per person (sent):")
for agent in network.agents:
    sent = network.get_message_count(source=agent)
    received = network.get_message_count(target=agent)
    print(f"  {agent.name}: sent {sent}, received {received}")

print("\n--- Interaction Summary ---")
print(network.pretty_current_interactions(simplified=True, max_content_length=200, last_n=5))

## Key Takeaways

This example demonstrates phenomena that **cannot be modeled with a basic `TinyWorld`**:

1. **Information Bottlenecks**: The CTO and VP of Product serve as information brokers. 
   Without them, R&D ideas never reach the sales team.

2. **Message Transformation**: As ideas pass through each layer, they get reframed — 
   technical details may be lost while business impact gets emphasized.

3. **Structural Influence**: Network position (measured by betweenness centrality) 
   determines who has the most influence over which ideas survive.

### Try It Yourself
- Change the hierarchy structure (e.g., flatter organization)
- Add more cross-functional links to see how it changes information flow
- Use `TinySocialNetworkFactory.create_department_network()` for a department-based structure
- Increase the number of simulation steps to see longer discussions